In [1]:
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql import SparkSession
from pyspark import SparkConf
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
    .master("spark://spark-master:7077")\
    .appName("jupyter-spark")\
    .config("spark.eventLog.enabled", "false")\
    .config("spark.driver.bindAddress", "0.0.0.0")\
    .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/04 10:05:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
actor_df = spark.read.csv('../data/actor.csv', header=True, inferSchema=True)
address_df = spark.read.csv('../data/address.csv', header=True, inferSchema=True)
category_df = spark.read.csv('../data/category.csv', header=True, inferSchema=True)
city_df = spark.read.csv('../data/city.csv', header=True, inferSchema=True)
country_df = spark.read.csv('../data/country.csv', header=True, inferSchema=True)
customer_df = spark.read.csv('../data/customer.csv', header=True, inferSchema=True)
film_df = spark.read.csv('../data/film.csv', header=True, inferSchema=True)
film_actor_df = spark.read.csv('../data/film_actor.csv', header=True, inferSchema=True)
film_category_df = spark.read.csv('../data/film_category.csv', header=True, inferSchema=True)
inventory_df = spark.read.csv('../data/inventory.csv', header=True, inferSchema=True)
language_df = spark.read.csv('../data/language.csv', header=True, inferSchema=True)
payment_df = spark.read.csv('../data/payment.csv', header=True, inferSchema=True)
rental_df = spark.read.csv('../data/rental.csv', header=True, inferSchema=True)
staff_df = spark.read.csv('../data/staff.csv', header=True, inferSchema=True)
store_df = spark.read.csv('../data/store.csv', header=True, inferSchema=True)

# Домашнє завдання на тему Spark SQL

Задачі з домашнього завдання на SQL потрібно розвʼязати за допомогою Spark SQL DataFrame API.

- Дампи таблиць знаходяться в папці `data`. Датафрейми таблиць вже створені в клітинці вище.
- Можете створювати стільки нових клітинок, скільки вам необхідно.
- Розвʼязок кожної задачі має бути відображений в самому файлі (використати метод `.show()`)
- код має бути оформлений у відповідності із одним із стилем, показаним лектором на занятті 13.

**Увага!**
Використовувати мову запитів SQL безпосередньо забороняється, потрібно використовувати виключно DataFrame API!


1.
Вивести кількість фільмів в кожній категорії.
Результат відсортувати за спаданням.

In [4]:
# Рахуємо кількість фільмів в кожній категорії
result_1 = film_category_df.join(
    category_df,
    film_category_df.category_id == category_df.category_id
).groupBy(
    "name"  # Назва категорії
).count().orderBy("count", ascending=False)

result_1.show()

[Stage 33:>                                                         (0 + 1) / 1]

+-----------+-----+
|       name|count|
+-----------+-----+
|     Sports|   74|
|    Foreign|   73|
|     Family|   69|
|Documentary|   68|
|  Animation|   66|
|     Action|   64|
|        New|   63|
|      Drama|   62|
|      Games|   61|
|     Sci-Fi|   61|
|   Children|   60|
|     Comedy|   58|
|     Travel|   57|
|   Classics|   57|
|     Horror|   56|
|      Music|   51|
+-----------+-----+



2.
Вивести 10 акторів, чиї фільми брали на прокат найбільше.
Результат відсортувати за спаданням.

In [7]:
from pyspark.sql.functions import concat_ws

# Join actor -> film_actor
actor_film_df = actor_df.join(film_actor_df, "actor_id")

# Join with inventory to get inventory_id
actor_film_inventory_df = actor_film_df.join(inventory_df, "film_id")

# Join with rental to get the rentals
actor_film_rental_df = actor_film_inventory_df.join(rental_df, "inventory_id")

# Групуємо по актору і рахуємо кількість оренд
result_2 = actor_film_rental_df.groupBy(
    concat_ws(" ", actor_df.first_name, actor_df.last_name).alias("actor_name")
).count().orderBy("count", ascending=False)

# Вивести топ 10
result_2.show(10, truncate=False)

[Stage 37:>                                                         (0 + 1) / 1]

+------------------+-----+
|actor_name        |count|
+------------------+-----+
|SUSAN DAVIS       |825  |
|GINA DEGENERES    |753  |
|MATTHEW CARREY    |678  |
|MARY KEITEL       |674  |
|ANGELA WITHERSPOON|654  |
|WALTER TORN       |640  |
|HENRY BERRY       |612  |
|JAYNE NOLTE       |611  |
|VAL BOLGER        |605  |
|SANDRA KILMER     |604  |
+------------------+-----+
only showing top 10 rows



3.
Вивести категорія фільмів, на яку було витрачено найбільше грошей
в прокаті

In [8]:
from pyspark.sql.functions import sum as _sum

# З'єднати фільм з категорією
film_category_joined_df = film_category_df.join(category_df, "category_id")

# З'єднати inventory
film_inventory_joined_df = film_category_joined_df.join(inventory_df, "film_id")

# З'єднати rental
film_rental_joined_df = film_inventory_joined_df.join(rental_df, "inventory_id")

# З'єднати payment
film_payment_joined_df = film_rental_joined_df.join(payment_df, "rental_id")

# Групуємо по категорії і сумуємо payment_amount
result_3 = film_payment_joined_df.groupBy(
    "name"  # Назва категорії
).agg(
    _sum("amount").alias("total_amount")
).orderBy("total_amount", ascending=False)

# Вивести 1 категорію (з найбільшими витратами)
result_3.show(1, truncate=False)

+------+-----------------+
|name  |total_amount     |
+------+-----------------+
|Sports|5314.209999999843|
+------+-----------------+
only showing top 1 row



4.
Вивести назви фільмів, яких не має в inventory.

In [9]:
# Вибираємо ті фільми, яких немає в inventory
result_4 = film_df.join(
    inventory_df,
    on="film_id",
    how="left_anti"   # left_anti = залишити тільки ті, що НЕ знайшли match
).select("title")

result_4.show(truncate=False)

[Stage 48:>                                                         (0 + 1) / 1]

+----------------------+
|title                 |
+----------------------+
|ALICE FANTASIA        |
|APOLLO TEEN           |
|ARGONAUTS TOWN        |
|ARK RIDGEMONT         |
|ARSENIC INDEPENDENCE  |
|BOONDOCK BALLROOM     |
|BUTCH PANTHER         |
|CATCH AMISTAD         |
|CHINATOWN GLADIATOR   |
|CHOCOLATE DUCK        |
|COMMANDMENTS EXPRESS  |
|CROSSING DIVORCE      |
|CROWDS TELEMARK       |
|CRYSTAL BREAKING      |
|DAZED PUNK            |
|DELIVERANCE MULHOLLAND|
|FIREHOUSE VIETNAM     |
|FLOATS GARDEN         |
|FRANKENSTEIN STRANGER |
|GLADIATOR WESTWARD    |
+----------------------+
only showing top 20 rows



5.
Вивести топ 3 актори, які найбільше зʼявлялись в категорії фільмів “Children”

In [10]:
from pyspark.sql.functions import col, concat_ws

# Вибираємо категорію "Children"
children_category_df = category_df.filter(col("name") == "Children")

# Джойнимо film_category з категоріями "Children"
children_film_df = film_category_df.join(
    children_category_df,
    on="category_id",
    how="inner"
)

# Джойнимо з film_actor щоб знайти акторів, які знімалися у цих фільмах
actor_children_df = children_film_df.join(
    film_actor_df,
    on="film_id",
    how="inner"
)

# Джойнимо з actor_df щоб отримати імена акторів
actor_names_df = actor_children_df.join(
    actor_df,
    on="actor_id",
    how="inner"
)

# Групуємо по імені актора і рахуємо кількість фільмів
result_5 = actor_names_df.groupBy(
    concat_ws(" ", "first_name", "last_name").alias("actor_name")
).count().orderBy(
    "count", ascending=False
)

# Виводимо топ-3
result_5.show(3, truncate=False)


[Stage 52:>                                                         (0 + 1) / 1]

+------------+-----+
|actor_name  |count|
+------------+-----+
|HELEN VOIGHT|7    |
|SUSAN DAVIS |6    |
|MARY TANDY  |5    |
+------------+-----+
only showing top 3 rows



Stop Spark session:

In [11]:
spark.stop()